In [11]:
import requests
import json
# Define the robot's Jupyter API URL
robot_ip = "192.168.0.102"  # Change to the robot's IP address
url = f"http://{robot_ip}:48888/api/kernels"
# Get the active kernel ID
response = requests.get(url)
kernels = response.json()
if not kernels:
    print("No active kernel found.")
    exit()
kernel_id = kernels[0]["id"]  # Assuming the first kernel is the active one
# Code to run on the robot (Modify as needed)
robot_code = """
from opentrons import protocol_api
metadata = {'apiLevel': '2.12'}
protocol = protocol_api.ProtocolContext()
pipette = protocol.load_instrument('p300_single', 'right')
pipette.home()
"""
# Send the code execution request
execute_url = f"http://{robot_ip}:48888/api/kernels/{kernel_id}/execute"
payload = {
    "code": robot_code,
    "silent": False
}
headers = {"Content-Type": "application/json"}
response = requests.post(execute_url, data=json.dumps(payload), headers=headers)
print(response)  # Check the response

<Response [404]>


In [8]:
response = requests.get(f"http://{robot_ip}:48888/api/kernels")
print(response.status_code,response.text)

200 [{"id": "97b12128-ee9d-4d11-a48a-c7b3740133a3", "name": "python3", "last_activity": "2025-03-07T01:08:08.624734Z", "execution_state": "idle", "connections": 0}, {"id": "6a85a3b9-e9bc-4e48-88c4-3ec4be9757de", "name": "python3", "last_activity": "2025-03-07T01:25:11.811483Z", "execution_state": "starting", "connections": 0}, {"id": "47d60b88-f17b-4c39-8eed-b8c1f2bc3e17", "name": "python3", "last_activity": "2025-03-07T01:26:53.472388Z", "execution_state": "starting", "connections": 0}, {"id": "4dd30fdf-9135-46b1-9e3b-b10deeacb3e5", "name": "python3", "last_activity": "2025-03-07T01:28:30.923899Z", "execution_state": "starting", "connections": 0}, {"id": "793e2c89-b797-44d1-b1c2-ec157ef96ee8", "name": "python3", "last_activity": "2025-03-07T01:28:36.866301Z", "execution_state": "starting", "connections": 0}, {"id": "530ae265-64a2-46b7-bd7d-c70b00a41e60", "name": "python3", "last_activity": "2025-03-07T01:28:40.533490Z", "execution_state": "starting", "connections": 0}, {"id": "f151697

In [12]:

response = requests.get(url)
kernels = response.json()
# Check if there are active kernels
if not kernels:
    print("No active kernel found. Try running a cell in the robot's Jupyter Notebook manually.")
    exit()
# Extract the first available kernel ID
kernel_id = kernels[0]["id"]
print(f"Active Kernel ID: {kernel_id}")

Active Kernel ID: 97b12128-ee9d-4d11-a48a-c7b3740133a3


In [23]:
execute_url = f"http://{robot_ip}:48888/api/kernels/{kernel_id}/channels"
robot_code = """
#Imports
from opentrons import protocol_api
from opentrons.types import Location, Point


##Need to write the metadata which is the info about the protocol, who wrote it, what version was it written in etc...
metadata = {
    #'apiLevel': '2.15', ##API version 
    'protocolName': 'Checking location_Map1', #Protocol name
    'description': '''This protocol will check the location if the pipet 
                    goes to the correct location.''',
    'author': 'Clara'
    }

##Recommended to have a "Requirements" section
requirements = {"robotType": "OT-2", "apiLevel": "2.19"} ##Note: If putting 'apilevel' here, remove it from the metadata

##Defining locations - map
#x = 125.3 + 10.00
#y = 86.88 + 3.00

##defining location 
x5 = 118.5 + 13.3    ## x5 & y5 are the coordinates for the bottom left corner of plate 5, same idea for x2 & y2 
y5 = 79.3 + 12.3    
x2 = 118.5 + 13.3    ## x_code = x_irl + 13.3
y2 = -11 + 12.3      ## y_code = y_irl + 12.3 

## width of box measured from middle of dividers = 132.4
## height of box measured from middle of dividers = 90.3


## positions of exact center points 
p0 = [[x2 + 0.125*132.4, y2 + 0.75*90.3],
      [x2 + 0.375*132.4, y2 + 0.75*90.3],
      [x2 + 0.625*132.4, y2 + 0.75*90.3],
      [x2 + 0.875*132.4, y2 + 0.75*90.3],
      [x2 + 0.125*132.4, y2 + 0.25*90.3],
      [x2 + 0.375*132.4, y2 + 0.25*90.3],
      [x2 + 0.625*132.4, y2 + 0.25*90.3],
      [x2 + 0.875*132.4, y2 + 0.25*90.3]]

p1 = [x5+0.25*132.4, y5 + 0.75*90.3]
p2 = [x5 + 0.75*132.4,p1[1]]
p3 = [p1[0],y5 + 0.25*90.3]
p4 = [p2[0],p3[1]]

def run(ctx):
    ''' input x, y, z values here (in mm). 
    Location of ice tray x = ,y = ,z = 

    '''
    #defining location of custom stuff
    loc0_0 = Location(Point(p0[0][0], p0[0][1], 6), None) # dispense - location of mixing
    loc0_1 = Location(Point(p0[1][0], p0[1][1], 6), None) # dispense - location of mixing
    loc1 = Location(Point(p1[0],p1[1],6),None) # location of blue dye 
    loc2 = Location(Point(p2[0],p2[1],6),None) # location of red dye 
    loc3 = Location(Point(p3[0],p3[1],6),None) # location of yellow dye
    loc4 = Location(Point(p4[0],p4[1],6),None)  # location of green dye

    #load labwear
    tiprack = ctx.load_labware('opentrons_96_tiprack_300ul', '1')
    pip = ctx.load_instrument('p300_single_gen2', 'right', tip_racks=[tiprack])

    #protocol for just moving the tip (for checking positions)
    for i in range(len(p0)):
        pip.move_to(Location(Point(p0[i][0], p0[i][1], 6), None))

    pip.move_to(loc1)
    
    pip.move_to(loc2)
    
    pip.move_to(loc3)
    
    pip.move_to(loc4)
   
    pip.home
"""

payload = {"code":robot_code,"silent":False}
headers = {"Content-Type":"application/json"}

response = requests.post(execute_url,data=json.dumps(payload),headers=headers)

print(response.status_code,response)

405 <Response [405]>
